# Testar o leitor de CRLV-e

Este notebook usa o pacote **extrator_dados_crlv** deste projeto para extrair **placa** e **chassi** de PDFs do CRLV-e.

**Como usar**

1. Coloque os PDFs na pasta do projeto (a mesma pasta deste notebook).
2. Execute as células em ordem com `Shift + Enter`.
3. Para testar outro arquivo, altere `ARQUIVO` na célula de seleção.

> Se não houver nenhum PDF na pasta, o notebook gera automaticamente um
> CRLV sintético com dados fictícios, para funcionar em um clone limpo do repositório.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import json

from IPython.display import Markdown, display

from extrator_dados_crlv import extrair_dados, extrair_dados_base64

PASTA = Path.cwd()
pdfs = sorted(PASTA.glob("*.pdf"))

# Para o notebook funcionar em um clone limpo, gera um CRLV sintético
# com dados fictícios quando não há PDF na pasta.
if not pdfs:
    import pymupdf

    campos = [
        ("CÓDIGO RENAVAM", "12345678901", 31, 90),
        ("PLACA", "ABC1D23", 31, 130),
        ("EXERCÍCIO", "2026", 103, 130),
        ("ANO FABRICAÇÃO", "2022", 31, 190),
        ("ANO MODELO", "2023", 103, 190),
        ("NÚMERO DO CRV", "123456789012", 31, 250),
        ("MARCA / MODELO / VERSÃO", "MARCA/TESTE 1.0", 31, 310),
        ("ESPÉCIE / TIPO", "MISTO UTILITARIO", 31, 370),
        ("PLACA ANTERIOR / UF", "XYZ9Z99/CE", 31, 430),
        ("CHASSI", "9BRK3AAG9P0000000", 160, 430),
        ("COR PREDOMINANTE", "PRATA", 31, 490),
        ("COMBUSTÍVEL", "ALCOOL/GASOLINA", 103, 490),
        ("CATEGORIA", "PARTICULAR", 317, 90),
        ("LOCAL", "FORTALEZA CE", 317, 250),
        ("DATA", "01/01/2026", 510, 250),
        ("NOME", "EMPRESA TESTE LTDA", 317, 190),
        ("CPF / CNPJ", "00.000.000/0001-00", 463, 220),
    ]

    destino = PASTA / "crlv_sintetico.pdf"
    with pymupdf.open() as documento:
        pagina = documento.new_page(width=595, height=842)
        for rotulo, valor, x, y in campos:
            pagina.insert_text((x, y), rotulo, fontsize=8)
            pagina.insert_text((x, y + 15), valor, fontsize=10)
        documento.save(destino)

    pdfs = [destino]
    print("Nenhum PDF encontrado; gerado crlv_sintetico.pdf com dados fictícios.")

print(f"Pasta: {PASTA}")
print(f"PDFs disponíveis: {len(pdfs)}")
for pdf in pdfs:
    print(" -", pdf.name)

In [ ]:
if not pdfs:
    raise FileNotFoundError("Coloque um PDF nesta pasta ou ajuste PASTA/ARQUIVO.")

# Troque pelo nome do arquivo que deseja testar, se necessário.
ARQUIVO = pdfs[0]

dados = extrair_dados(ARQUIVO)
display(Markdown(f"### Placa: `{dados.placa}`\n### Chassi: `{dados.chassi}`"))

In [ ]:
print(json.dumps(asdict(dados), ensure_ascii=False, indent=2))

## Testar a partir de base64

Simula o cenário em que o PDF chega pela API como uma string base64, em vez de um arquivo.

In [ ]:
import base64

conteudo_b64 = base64.b64encode(ARQUIVO.read_bytes()).decode("ascii")
print(f"Base64 gerado: {len(conteudo_b64):,} caracteres")

dados_b64 = extrair_dados(conteudo_b64, base64=True)
print("Placa: ", dados_b64.placa)
print("Chassi:", dados_b64.chassi)

assert (dados_b64.placa, dados_b64.chassi) == (dados.placa, dados.chassi)
print("OK: resultado igual ao da leitura por arquivo.")

In [ ]:
# Também aceita data URI, formato comum em APIs.
data_uri = "data:application/pdf;base64," + conteudo_b64
dados_uri = extrair_dados_base64(data_uri)
print("Data URI:", dados_uri.placa, dados_uri.chassi)

## Testar todos os PDFs da pasta

In [ ]:
resultados = []
for pdf in pdfs:
    try:
        item = extrair_dados(pdf)
        resultados.append(
            {"arquivo": pdf.name, "placa": item.placa, "chassi": item.chassi, "erro": ""}
        )
    except Exception as erro:
        resultados.append(
            {"arquivo": pdf.name, "placa": None, "chassi": None, "erro": str(erro)}
        )

for item in resultados:
    status = item["erro"] or "OK"
    print(f"{item['arquivo']:<45} placa={item['placa']}  chassi={item['chassi']}  {status}")

## Validação rápida

A célula abaixo confirma que placa e chassi foram encontrados no arquivo selecionado.

In [ ]:
assert dados.placa, "Placa não encontrada"
assert dados.chassi, "Chassi não encontrado"
assert len(dados.chassi) == 17, "Chassi não tem 17 caracteres"

print("OK: placa e chassi válidos.")

## Depuração

Mostra as linhas do PDF que contêm os rótulos ou os valores extraídos, com as coordenadas usadas pelo extrator.

In [ ]:
import pymupdf

from extrator_dados_crlv.crlv import _ler_linhas

termos = {"PLACA", "CHASSI"}
if dados.placa:
    termos.add(dados.placa)
if dados.chassi:
    termos.add(dados.chassi)

with pymupdf.open(ARQUIVO) as documento:
    linhas = _ler_linhas(documento)

for linha in linhas:
    if any(termo in linha.texto.upper() for termo in termos):
        print(
            f"pág {linha.pagina + 1} | x={linha.x0:6.1f} y={linha.y0:6.1f} | {linha.texto}"
        )

## Testes automatizados

Executa a suíte de testes do projeto.

In [ ]:
!uv run python -m unittest discover -s tests -v